# CorrelationSelector

Ranks features by the absolute correlation with the target, then removes
redundant copies.

1. Correlate every feature with the target.
2. Drop features below `target_threshold`.
3. Of any pair correlated above `inter_feature_threshold`, keep the one closer
   to the target.
4. Keep the top `n_features`, if given.

- **Strength:** the only selector here that removes redundancy explicitly.
- **Weakness:** linear (or rank, with `method='spearman'`) relationships only;
  the pairwise step grows with the square of the candidate count.
- The target must be numeric, so it suits regression or a 0/1 class.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
from featsel.selectors import CorrelationSelector

n = 300
signal = rng.normal(size=(n, 3))
y = signal @ np.array([3.0, 2.0, 1.0]) + rng.normal(size=n)
X = pd.DataFrame(np.column_stack([signal,
                                  signal[:, 0] + rng.normal(0, 0.05, n),  # near-copy of s0
                                  rng.normal(size=(n, 6))]),
                 columns=['s0', 's1', 's2', 's0_copy'] + [f'noise{i}' for i in range(6)])
X.corrwith(pd.Series(y)).abs().round(2)

s0         0.77
s1         0.54
s2         0.22
s0_copy    0.77
noise0     0.09
noise1     0.04
noise2     0.07
noise3     0.04
noise4     0.04
noise5     0.05
dtype: float64

## Redundancy removal

In [2]:
redundant = CorrelationSelector(target_threshold=0.2, inter_feature_threshold=0.95).fit(X, y)
print('kept:', redundant.selected_features_)

kept: ['s0', 's1', 's2']


`s0_copy` is almost as correlated with the target as `s0`, but it adds
nothing once `s0` is in. Loosening the pair threshold keeps both.

In [3]:
both = CorrelationSelector(target_threshold=0.2, inter_feature_threshold=0.999).fit(X, y)
print('kept:', both.selected_features_)

kept: ['s0', 's1', 's2', 's0_copy']
